# Manual Corrective Interventions

Targeted top-1 repair-rate experiment for misclassified test videos where the ground-truth class is already in the model top-5. Run the notebook top to bottom, then use the widget to annotate up to four edits per candidate. Each click tests prefixes with k = 1..4 and logs the first k that repairs the top-1 prediction.


## 1. Configuration

Change only this cell to switch dataset/model/checkpoint. `model_path = None` uses the default checkpoint naming convention.


In [ ]:
from pathlib import Path

CONFIG = {
    # Model/data identity.
    "dataset": "breakfast",          # breakfast, ucf101, hmdb51, something2
    "dataset_name": "Breakfast",     # Breakfast, UCF101, HMDB, Something2
    "clip_model": "res50",           # res50, b16, b32, l14, siglip, clip4clip, pe-l14, ...
    "window_size": 32,
    "test_split": "s1",

    # Optional explicit checkpoint. If None, auto-resolve from the fields above.
    "model_path": None,

    # Runtime.
    "device": "cuda:0",              # use "cpu" if CUDA is busy/unavailable
    "seed": 42,

    # Experiment size.
    "n_samples": 100,
    "min_samples": 100,
    "max_samples": 100,
    "max_concepts_per_video": 4,
    # Manual-edit handling.
    "manual_annotation_csv": "manual_corrective_annotations.csv",
    "topk_filter": 5,
}

ROOT = Path.cwd()
if ROOT.name != "NeurIPS26_MOTIF_supplement":
    # This keeps relative paths stable when the notebook is launched from elsewhere.
    ROOT = Path(".")

if CONFIG["model_path"] is None:
    CONFIG["model_path"] = str(
        ROOT / "Models" / f"{CONFIG['dataset_name']}_{CONFIG['clip_model']}_testing_1.0_{CONFIG['window_size']}_motif.pkl"
    )

CONFIG


## 2. Imports And Helpers

These helpers are local to the notebook. The checkpoint is loaded with a CPU-safe fallback, then the model is moved to the configured device.


In [ ]:
import io
import os
import sys
import math
import pickle
import warnings
import contextlib
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Video, display

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "utils"))

from utils.motif import init_repro
from utils.explanations import explain_instance

seed = init_repro(CONFIG["seed"], deterministic=True)


def resolve_device(device_name: str) -> torch.device:
    if device_name.startswith("cuda") and not torch.cuda.is_available():
        warnings.warn("CUDA was requested but is not available; using CPU.")
        return torch.device("cpu")
    return torch.device(device_name)


@contextlib.contextmanager
def torch_pickle_map_location(map_location="cpu"):
    """Force legacy torch storages inside pickle files onto a safe device."""
    old_loader = torch.storage._load_from_bytes

    def mapped_loader(storage_bytes):
        return torch.load(
            io.BytesIO(storage_bytes),
            map_location=map_location,
            weights_only=False,
        )

    torch.storage._load_from_bytes = mapped_loader
    try:
        yield
    finally:
        torch.storage._load_from_bytes = old_loader


def load_cbm_checkpoint(path: str, device: torch.device):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    with torch_pickle_map_location("cpu"):
        with path.open("rb") as f:
            cbm_model = pickle.load(f)
    if getattr(cbm_model, "model", None) is None:
        raise ValueError("Loaded checkpoint does not contain cbm_model.model")
    try:
        cbm_model.model = cbm_model.model.to(device).eval()
        cbm_model._notebook_device = device
    except Exception as exc:
        if device.type != "cuda":
            raise
        warnings.warn(f"Could not move model to {device}: {exc}. Falling back to CPU.")
        fallback = torch.device("cpu")
        cbm_model.model = cbm_model.model.to(fallback).eval()
        cbm_model._notebook_device = fallback
    return cbm_model


def prepare_video(video, device: torch.device):
    x = video if torch.is_tensor(video) else torch.as_tensor(np.asarray(video), dtype=torch.float32)
    x = x.to(device=device, dtype=torch.float32)
    if x.dim() == 2:
        x = x.unsqueeze(0)
    return x


def label_name(cbm_model, idx: int) -> str:
    try:
        return str(cbm_model.encoder.inverse_transform([int(idx)])[0])
    except Exception:
        return str(idx)


def concept_names_for_model(cbm_model) -> List[str]:
    n = int(getattr(cbm_model, "num_concepts", cbm_model.X_test[0].shape[-1]))
    names = None
    if getattr(cbm_model, "concepts", None) is not None:
        names = getattr(cbm_model.concepts, "text_concepts", None)
    if names is None:
        names = getattr(cbm_model, "text_concepts", None)
    if names is None or len(names) != n:
        return [f"c{i}" for i in range(n)]
    return [str(x) for x in names]


def class_rank(logits: torch.Tensor, class_idx: int) -> int:
    logits = logits.detach().flatten().cpu()
    order = torch.argsort(logits, descending=True)
    return int((order == int(class_idx)).nonzero(as_tuple=True)[0].item() + 1)


def logits_to_record(cbm_model, sample_idx: int, logits: torch.Tensor) -> Dict[str, object]:
    logits = logits.detach().flatten().cpu()
    true_idx = int(cbm_model.y_test[sample_idx])
    pred_idx = int(torch.argmax(logits).item())
    return {
        "sample_idx": int(sample_idx),
        "true_idx": true_idx,
        "pred_idx": pred_idx,
        "true_label": label_name(cbm_model, true_idx),
        "pred_label": label_name(cbm_model, pred_idx),
        "true_rank": class_rank(logits, true_idx),
        "true_logit": float(logits[true_idx].item()),
        "pred_logit": float(logits[pred_idx].item()),
        "top5": [int(i) for i in torch.argsort(logits, descending=True)[:5].tolist()],
    }


@torch.no_grad()
def forward_logits(cbm_model, sample_idx: int, device: torch.device) -> torch.Tensor:
    x = prepare_video(cbm_model.X_test[sample_idx], device)
    logits, _, _, _ = cbm_model.model(x)
    return logits[0].detach().cpu()


@torch.no_grad()
def compute_concepts_t(model, video: torch.Tensor, key_padding_mask=None) -> torch.Tensor:
    """Replay CBMTransformer.forward until concepts_t, before concept interventions."""
    x = video
    if x.dim() == 2:
        x = x.unsqueeze(0)
        if key_padding_mask is not None and key_padding_mask.dim() == 1:
            key_padding_mask = key_padding_mask.unsqueeze(0)

    x = model.posenc(x)
    for layer in model.layers:
        if getattr(model, "diagonal_attention", False):
            x = layer(x, key_padding_mask=key_padding_mask)
        else:
            x = layer(x, key_padding_mask=key_padding_mask)
    x = model.norm(x)
    return model.concept_predictor(x)


@torch.no_grad()
def pool_logits_from_concepts_t(model, concepts_t: torch.Tensor, key_padding_mask=None) -> torch.Tensor:
    logits_t = model.classifier(concepts_t)
    tau = float(model.lse_tau)
    if key_padding_mask is not None:
        logits_t = logits_t.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
    return (logits_t * tau).logsumexp(dim=1) / tau


@torch.no_grad()
def logits_with_concept_edits(
    cbm_model,
    sample_idx: int,
    edits: Sequence[Tuple[int, int, float]],
    device: torch.device,
) -> torch.Tensor:
    model = cbm_model.model
    x = prepare_video(cbm_model.X_test[sample_idx], device)
    concepts_t = compute_concepts_t(model, x).clone()
    _, num_windows, num_concepts = concepts_t.shape

    for window_idx, concept_idx, value in edits:
        w = int(window_idx)
        c = int(concept_idx)
        if 0 <= w < num_windows and 0 <= c < num_concepts and pd.notna(value):
            concepts_t[:, w, c] = float(value)

    return pool_logits_from_concepts_t(model, concepts_t)[0].detach().cpu()


## 3. Load Checkpoint


In [ ]:
device = resolve_device(CONFIG["device"])
cbm_model = load_cbm_checkpoint(CONFIG["model_path"], device)
device = getattr(cbm_model, "_notebook_device", device)
concept_names = concept_names_for_model(cbm_model)

print(f"Loaded: {CONFIG['model_path']}")
print(f"Device: {device}")
print(f"Test videos: {len(cbm_model.X_test)}")
print(f"Classes: {len(getattr(cbm_model.encoder, 'classes_', []))}")
print(f"Concepts: {len(concept_names)}")


## 4. Select Repair Candidates

Candidates are misclassified test videos whose true label is already within the top-5 logits.


In [ ]:
def select_repair_candidates(cbm_model, device: torch.device, config: Dict[str, object]) -> pd.DataFrame:
    rows = []
    model = cbm_model.model.eval()
    n_total = len(cbm_model.X_test)

    for sample_idx in range(n_total):
        logits = forward_logits(cbm_model, sample_idx, device)
        rec = logits_to_record(cbm_model, sample_idx, logits)
        if rec["pred_idx"] != rec["true_idx"] and rec["true_rank"] <= int(config["topk_filter"]):
            rec["video_path"] = (
                cbm_model.paths_test[sample_idx]
                if getattr(cbm_model, "paths_test", None) is not None and sample_idx < len(cbm_model.paths_test)
                else None
            )
            rows.append(rec)

    candidates = pd.DataFrame(rows)
    if candidates.empty:
        warnings.warn("No eligible repair candidates found.")
        return candidates

    n_requested = min(int(config["n_samples"]), int(config["max_samples"]))
    if len(candidates) < int(config["min_samples"]):
        warnings.warn(
            f"Only {len(candidates)} eligible candidates found; requested minimum is {config['min_samples']}."
        )
    if len(candidates) > n_requested:
        candidates = candidates.sample(n=n_requested, random_state=int(config["seed"])).sort_values("sample_idx")

    return candidates.reset_index(drop=True)


candidates_df = select_repair_candidates(cbm_model, device, CONFIG)
print(f"Selected {len(candidates_df)} candidates")
if not candidates_df.empty:
    display(candidates_df[["sample_idx", "true_label", "pred_label", "true_rank"]].head(10))


## 5. Build Suggested Edit Tables

Local edits are cell-wise `(window, concept)` interventions. Global edits are column-wise concept interventions applied across all temporal windows. Both tables contain up to `max_concepts_per_video` suggestions per candidate; the widget lets you override values manually.


In [ ]:
@torch.no_grad()
def suggested_value_for_slot(concepts_t: torch.Tensor, window_idx: int, concept_idx: int, true_idx: int, pred_idx: int, model) -> float:
    values = concepts_t[:, int(concept_idx)].detach().cpu().float()
    current = float(values[int(window_idx)].item())
    w_true = float(model.classifier.weight[int(true_idx), int(concept_idx)].detach().cpu().item())
    w_pred = float(model.classifier.weight[int(pred_idx), int(concept_idx)].detach().cpu().item())
    delta_weight = w_true - w_pred

    if values.numel() <= 1:
        return current + (1.0 if delta_weight >= 0 else -1.0)

    q = 0.90 if delta_weight >= 0 else 0.10
    proposed = float(torch.quantile(values, torch.tensor(q)).item())
    if math.isclose(proposed, current, rel_tol=1e-6, abs_tol=1e-6):
        proposed = current + (1.0 if delta_weight >= 0 else -1.0)
    return proposed


@torch.no_grad()
def suggested_value_for_global_concept(concepts_t: torch.Tensor, concept_idx: int, true_idx: int, pred_idx: int, model) -> float:
    values = concepts_t[:, int(concept_idx)].detach().cpu().float()
    if values.numel() == 0:
        return 0.0
    w_true = float(model.classifier.weight[int(true_idx), int(concept_idx)].detach().cpu().item())
    w_pred = float(model.classifier.weight[int(pred_idx), int(concept_idx)].detach().cpu().item())
    delta_weight = w_true - w_pred
    q = 0.90 if delta_weight >= 0 else 0.10
    proposed = float(torch.quantile(values, torch.tensor(q)).item())
    current = float(values.mean().item())
    if math.isclose(proposed, current, rel_tol=1e-6, abs_tol=1e-6):
        proposed = current + (1.0 if delta_weight >= 0 else -1.0)
    return proposed


def build_local_edit_table(cbm_model, candidates: pd.DataFrame, device: torch.device, config: Dict[str, object]) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame()

    rows = []
    model = cbm_model.model.eval()
    max_rows = int(config["max_concepts_per_video"])

    for rec in candidates.to_dict("records"):
        sample_idx = int(rec["sample_idx"])
        true_idx = int(rec["true_idx"])
        pred_idx = int(rec["pred_idx"])
        video = prepare_video(cbm_model.X_test[sample_idx], device)

        exp_wrong = explain_instance(model, video, target_class=pred_idx)
        exp_true = explain_instance(model, video, target_class=true_idx)
        wrong = exp_wrong["concept_contributions_per_time"].detach().cpu().float()
        true = exp_true["concept_contributions_per_time"].detach().cpu().float()
        concepts_t = exp_wrong["concepts_per_time"].detach().cpu().float()
        priority = wrong - true

        flat = priority.reshape(-1)
        positive = torch.where(flat > 0, flat, torch.zeros_like(flat))
        ranked = torch.argsort(positive if int((positive > 0).sum().item()) > 0 else flat.abs(), descending=True)

        T, C = priority.shape
        used = set()
        for flat_idx in ranked.tolist():
            window_idx, concept_idx = np.unravel_index(flat_idx, (T, C))
            key = (int(window_idx), int(concept_idx))
            if key in used:
                continue
            used.add(key)
            suggested = suggested_value_for_slot(concepts_t, window_idx, concept_idx, true_idx, pred_idx, model)
            rows.append({
                "edit_mode": "local",
                "sample_idx": sample_idx,
                "video_path": rec.get("video_path"),
                "true_label": rec["true_label"],
                "pred_label": rec["pred_label"],
                "window_idx": int(window_idx),
                "concept_idx": int(concept_idx),
                "concept_name": concept_names[int(concept_idx)] if int(concept_idx) < len(concept_names) else f"c{concept_idx}",
                "true_rank_before": int(rec["true_rank"]),
                "wrong_contribution": float(wrong[int(window_idx), int(concept_idx)].item()),
                "true_contribution": float(true[int(window_idx), int(concept_idx)].item()),
                "repair_priority": float(priority[int(window_idx), int(concept_idx)].item()),
                "suggested_correction_value": float(suggested),
                "apply_edit": False,
                "correction_value": np.nan,
            })
            if len(used) >= max_rows:
                break

    return pd.DataFrame(rows).sort_values(["sample_idx", "repair_priority"], ascending=[True, False]).reset_index(drop=True)


def build_global_edit_table(cbm_model, candidates: pd.DataFrame, device: torch.device, config: Dict[str, object]) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame()

    rows = []
    model = cbm_model.model.eval()
    max_rows = int(config["max_concepts_per_video"])

    for rec in candidates.to_dict("records"):
        sample_idx = int(rec["sample_idx"])
        true_idx = int(rec["true_idx"])
        pred_idx = int(rec["pred_idx"])
        video = prepare_video(cbm_model.X_test[sample_idx], device)

        exp_pred = explain_instance(model, video, target_class=pred_idx)
        exp_true = explain_instance(model, video, target_class=true_idx)
        pred_global = exp_pred["concept_contributions_global"].detach().cpu().float()
        true_global = exp_true["concept_contributions_global"].detach().cpu().float()
        concepts_t = exp_pred["concepts_per_time"].detach().cpu().float()
        priority = pred_global - true_global

        positive = torch.where(priority > 0, priority, torch.zeros_like(priority))
        ranked = torch.argsort(positive if int((positive > 0).sum().item()) > 0 else priority.abs(), descending=True)

        used = set()
        for concept_idx in ranked.tolist():
            concept_idx = int(concept_idx)
            if concept_idx in used:
                continue
            used.add(concept_idx)
            suggested = suggested_value_for_global_concept(concepts_t, concept_idx, true_idx, pred_idx, model)
            rows.append({
                "edit_mode": "global",
                "sample_idx": sample_idx,
                "video_path": rec.get("video_path"),
                "true_label": rec["true_label"],
                "pred_label": rec["pred_label"],
                "window_idx": np.nan,
                "concept_idx": concept_idx,
                "concept_name": concept_names[concept_idx] if concept_idx < len(concept_names) else f"c{concept_idx}",
                "true_rank_before": int(rec["true_rank"]),
                "pred_contribution": float(pred_global[concept_idx].item()),
                "true_contribution": float(true_global[concept_idx].item()),
                "repair_priority": float(priority[concept_idx].item()),
                "suggested_correction_value": float(suggested),
                "apply_edit": False,
                "correction_value": np.nan,
            })
            if len(used) >= max_rows:
                break

    return pd.DataFrame(rows).sort_values(["sample_idx", "repair_priority"], ascending=[True, False]).reset_index(drop=True)


local_edit_df = build_local_edit_table(cbm_model, candidates_df, device, CONFIG)
global_edit_df = build_global_edit_table(cbm_model, candidates_df, device, CONFIG)
annotation_df = local_edit_df

print(f"Local edit suggestions: {len(local_edit_df)} rows")
print(f"Global edit suggestions: {len(global_edit_df)} rows")


In [ ]:
# Optional: save or reload the suggestion tables.
# local_edit_df.to_csv(ROOT / "manual_local_edit_suggestions.csv", index=False)
# global_edit_df.to_csv(ROOT / "manual_global_edit_suggestions.csv", index=False)
# local_edit_df = pd.read_csv(ROOT / "manual_local_edit_suggestions.csv")
# global_edit_df = pd.read_csv(ROOT / "manual_global_edit_suggestions.csv")


## 6. Inspect Videos And Concepts

Shows the video/frame strip, top-10 global concept plots, and a 2x2 plot of the top 10 concepts in the four most relevant temporal windows.


In [ ]:
def get_window_spans(cbm_model, sample_idx: int):
    path = None
    if getattr(cbm_model, "paths_test", None) is not None and sample_idx < len(cbm_model.paths_test):
        path = cbm_model.paths_test[sample_idx]
    if path is None:
        return None
    spans = None
    if getattr(cbm_model, "video_spans", None) is not None:
        spans = cbm_model.video_spans.get(path)
    if spans is None and getattr(cbm_model, "video_window_spans", None) is not None:
        spans = cbm_model.video_window_spans.get(path)
    return spans


def global_concept_table(sample_idx: int) -> pd.DataFrame:
    logits = forward_logits(cbm_model, sample_idx, device)
    rec = logits_to_record(cbm_model, sample_idx, logits)
    video = prepare_video(cbm_model.X_test[sample_idx], device)

    exp_pred = explain_instance(cbm_model.model, video, target_class=rec["pred_idx"])
    exp_true = explain_instance(cbm_model.model, video, target_class=rec["true_idx"])

    pred_global = exp_pred["concept_contributions_global"].detach().cpu().float()
    true_global = exp_true["concept_contributions_global"].detach().cpu().float()
    rows = []
    for concept_idx in range(len(pred_global)):
        rows.append({
            "sample_idx": int(sample_idx),
            "concept_idx": int(concept_idx),
            "concept_name": concept_names[concept_idx] if concept_idx < len(concept_names) else f"c{concept_idx}",
            "pred_label": rec["pred_label"],
            "true_label": rec["true_label"],
            "pred_contribution": float(pred_global[concept_idx].item()),
            "true_contribution": float(true_global[concept_idx].item()),
            "repair_priority": float((pred_global[concept_idx] - true_global[concept_idx]).item()),
        })
    return pd.DataFrame(rows)


def plot_global_important_concepts(sample_idx: int, top_k: int = 10, modes=("pred", "true", "repair_priority")) -> pd.DataFrame:
    table = global_concept_table(sample_idx)
    if table.empty:
        print("No global concepts to plot.")
        return table

    mode_to_column = {
        "pred": "pred_contribution",
        "true": "true_contribution",
        "repair_priority": "repair_priority",
        "abs_pred": "pred_contribution",
        "abs_true": "true_contribution",
    }
    titles = {
        "pred": "Concepts supporting predicted class",
        "true": "Concepts supporting true class",
        "repair_priority": "Wrong-vs-true repair priority",
        "abs_pred": "Predicted-class concepts by absolute contribution",
        "abs_true": "True-class concepts by absolute contribution",
    }
    colors = {
        "pred": "tab:red",
        "true": "tab:green",
        "repair_priority": "tab:orange",
        "abs_pred": "tab:red",
        "abs_true": "tab:green",
    }

    fig, axes = plt.subplots(1, len(modes), figsize=(5.2 * len(modes), max(3.2, 0.32 * top_k)))
    if len(modes) == 1:
        axes = [axes]

    plotted_parts = []
    for ax, mode in zip(axes, modes):
        column = mode_to_column[mode]
        if mode.startswith("abs_"):
            ranked = table.assign(_rank=table[column].abs()).sort_values("_rank", ascending=False).head(top_k)
        else:
            ranked = table.sort_values(column, ascending=False).head(top_k)
        ranked = ranked.iloc[::-1]
        labels = [f"c{int(r.concept_idx)} {r.concept_name}" for r in ranked.itertuples(index=False)]
        ax.barh(labels, ranked[column], color=colors.get(mode, "tab:blue"))
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(titles.get(mode, mode))
        ax.set_xlabel(column)
        plotted_parts.append(ranked.drop(columns=["_rank"], errors="ignore").assign(plot_mode=mode))

    plt.tight_layout()
    plt.show()
    return pd.concat(plotted_parts, ignore_index=True)


def local_concept_table(sample_idx: int) -> pd.DataFrame:
    logits = forward_logits(cbm_model, sample_idx, device)
    rec = logits_to_record(cbm_model, sample_idx, logits)
    video = prepare_video(cbm_model.X_test[sample_idx], device)

    exp_pred = explain_instance(cbm_model.model, video, target_class=rec["pred_idx"])
    exp_true = explain_instance(cbm_model.model, video, target_class=rec["true_idx"])

    pred_local = exp_pred["concept_contributions_per_time"].detach().cpu().float()
    true_local = exp_true["concept_contributions_per_time"].detach().cpu().float()
    priority = pred_local - true_local

    rows = []
    T, C = priority.shape
    for window_idx in range(T):
        for concept_idx in range(C):
            rows.append({
                "sample_idx": int(sample_idx),
                "window_idx": int(window_idx),
                "concept_idx": int(concept_idx),
                "concept_name": concept_names[concept_idx] if concept_idx < len(concept_names) else f"c{concept_idx}",
                "pred_label": rec["pred_label"],
                "true_label": rec["true_label"],
                "pred_contribution": float(pred_local[window_idx, concept_idx].item()),
                "true_contribution": float(true_local[window_idx, concept_idx].item()),
                "repair_priority": float(priority[window_idx, concept_idx].item()),
            })
    return pd.DataFrame(rows)


def plot_top_window_concepts(sample_idx: int, top_windows: int = 4, top_k: int = 10, rank_by: str = "repair_priority") -> pd.DataFrame:
    table = local_concept_table(sample_idx)
    if table.empty:
        print("No local concepts to plot.")
        return table

    window_scores = (
        table.assign(_positive=table[rank_by].clip(lower=0.0))
        .groupby("window_idx", as_index=False)["_positive"]
        .sum()
        .sort_values("_positive", ascending=False)
        .head(top_windows)
    )
    if float(window_scores["_positive"].sum()) == 0.0:
        window_scores = (
            table.assign(_abs=table[rank_by].abs())
            .groupby("window_idx", as_index=False)["_abs"]
            .sum()
            .sort_values("_abs", ascending=False)
            .head(top_windows)
        )
    selected_windows = [int(w) for w in window_scores["window_idx"].tolist()]

    ncols = 2
    nrows = int(np.ceil(max(1, len(selected_windows)) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, max(3.2, 3.0 * nrows)))
    axes = np.asarray(axes).reshape(-1)

    plotted_parts = []
    for ax, window_idx in zip(axes, selected_windows):
        part = table[table["window_idx"].eq(window_idx)].sort_values(rank_by, ascending=False).head(top_k).iloc[::-1]
        labels = [f"c{int(r.concept_idx)} {r.concept_name}" for r in part.itertuples(index=False)]
        ax.barh(labels, part[rank_by], color="tab:purple")
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"window {window_idx}: top {top_k} local concepts")
        ax.set_xlabel(rank_by)
        plotted_parts.append(part.assign(plot_window_rank=len(plotted_parts) + 1))

    for ax in axes[len(selected_windows):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return pd.concat(plotted_parts, ignore_index=True) if plotted_parts else pd.DataFrame()


def inspect_video(
    sample_idx: int,
    annotation_table: pd.DataFrame,
    num_frames: int = 8,
    show_global_concepts: bool = True,
    global_top_k: int = 10,
    window_top_k: int = 10,
    num_top_windows: int = 4,
):
    if getattr(cbm_model, "paths_test", None) is None:
        print("No paths_test found in checkpoint.")
        return
    if sample_idx >= len(cbm_model.paths_test):
        print(f"sample_idx {sample_idx} is outside paths_test.")
        return

    rec = logits_to_record(cbm_model, sample_idx, forward_logits(cbm_model, sample_idx, device))
    print(f"sample {sample_idx}: pred={rec['pred_label']} | true={rec['true_label']} | true rank={rec['true_rank']}")

    video_path = cbm_model.paths_test[sample_idx]
    video_path = video_path.replace("../", "../../Data/")
    print(video_path)
    if video_path and os.path.exists(video_path):
        display(Video(video_path, embed=False, width=480))
    else:
        print("Raw video file is unavailable; showing plots only.")
        if show_global_concepts:
            plot_global_important_concepts(sample_idx, top_k=global_top_k)
            plot_top_window_concepts(sample_idx, top_windows=num_top_windows, top_k=window_top_k)
        return

    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0) or 30.0
    if frame_count <= 0:
        cap.release()
        print("Could not read video frames.")
        if show_global_concepts:
            plot_global_important_concepts(sample_idx, top_k=global_top_k)
            plot_top_window_concepts(sample_idx, top_windows=num_top_windows, top_k=window_top_k)
        return

    spans = get_window_spans(cbm_model, sample_idx)
    annotated_windows = set()
    if annotation_table is not None and not annotation_table.empty and "window_idx" in annotation_table:
        annotated_windows = set(
            int(w) for w in annotation_table.loc[annotation_table["sample_idx"].eq(sample_idx), "window_idx"].dropna().tolist()
        )
    annotated_ranges = []
    if spans is not None:
        for w in annotated_windows:
            if 0 <= w < len(spans):
                annotated_ranges.append(tuple(spans[w]))

    frame_ids = np.linspace(0, frame_count - 1, num=min(num_frames, frame_count), dtype=int)
    fig, axes = plt.subplots(1, len(frame_ids), figsize=(2.2 * len(frame_ids), 2.4))
    if len(frame_ids) == 1:
        axes = [axes]

    for ax, frame_id in zip(axes, frame_ids):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
        ok, frame = cap.read()
        if not ok:
            ax.axis("off")
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        t_sec = frame_id / fps
        highlighted = any(start <= t_sec <= end for start, end in annotated_ranges)
        ax.imshow(frame)
        ax.set_title(f"{t_sec:.1f}s", color=("red" if highlighted else "black"))
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(highlighted)
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    cap.release()
    plt.tight_layout()
    plt.show()

    if show_global_concepts:
        plot_global_important_concepts(sample_idx, top_k=global_top_k)
        plot_top_window_concepts(sample_idx, top_windows=num_top_windows, top_k=window_top_k)


In [ ]:
# Manual inspection cell. Change the row number or sample_idx and run this cell once.
from IPython.display import clear_output
clear_output(wait=True)

row = 0
sample_idx = int(candidates_df.iloc[row]["sample_idx"])
inspect_video(sample_idx, local_edit_df)


## 7. Sequential Manual Repair UI

Use this for annotation, testing, and logging. The widget renders the current video, global concept plots, local window plots, and a text logit summary in controlled output panes so reruns do not duplicate notebook output.


In [ ]:
@torch.no_grad()
def logits_with_global_concept_edits(
    cbm_model,
    sample_idx: int,
    edits: Sequence[Tuple[int, float]],
    device: torch.device,
) -> torch.Tensor:
    model = cbm_model.model
    x = prepare_video(cbm_model.X_test[sample_idx], device)
    concepts_t = compute_concepts_t(model, x).clone()
    _, _, num_concepts = concepts_t.shape

    for concept_idx, value in edits:
        c = int(concept_idx)
        if 0 <= c < num_concepts and pd.notna(value):
            concepts_t[:, :, c] = float(value)

    return pool_logits_from_concepts_t(model, concepts_t)[0].detach().cpu()


@torch.no_grad()
def original_concepts_for_sample(cbm_model, sample_idx: int, device: torch.device) -> torch.Tensor:
    model = cbm_model.model
    x = prepare_video(cbm_model.X_test[sample_idx], device)
    return compute_concepts_t(model, x)[0].detach().cpu().float()


def concept_label(concept_idx: int) -> str:
    name = concept_names[concept_idx] if concept_idx < len(concept_names) else f"c{concept_idx}"
    return f"c{concept_idx} | {name}"


def parse_concept_label(value, label_to_idx: Dict[str, int]) -> int:
    text = str(value).strip()
    if text in label_to_idx:
        return int(label_to_idx[text])
    if text.startswith("c") and "|" in text:
        text = text.split("|", 1)[0].strip()[1:]
    if text.startswith("c") and text[1:].isdigit():
        text = text[1:]
    if text.isdigit():
        idx = int(text)
        if 0 <= idx < len(concept_names):
            return idx
    matches = [idx for label, idx in label_to_idx.items() if str(value).strip().lower() in label.lower()]
    if len(matches) == 1:
        return int(matches[0])
    raise ValueError(f"Unknown or ambiguous concept: {value}")


def plot_logit_before_after(cbm_model, base_logits: torch.Tensor, after_logits: torch.Tensor, base: Dict[str, object], after: Dict[str, object], title: str = ""):
    class_ids = []
    for idx in [base["pred_idx"], base["true_idx"], after["pred_idx"]]:
        idx = int(idx)
        if idx not in class_ids:
            class_ids.append(idx)
    labels = [label_name(cbm_model, idx) for idx in class_ids]
    before = [float(base_logits[int(idx)].item()) for idx in class_ids]
    after_values = [float(after_logits[int(idx)].item()) for idx in class_ids]
    x = np.arange(len(class_ids))
    width = 0.36
    fig, ax = plt.subplots(figsize=(max(5.0, 1.8 * len(class_ids)), 3.4))
    ax.bar(x - width / 2, before, width, label="before", color="tab:gray")
    ax.bar(x + width / 2, after_values, width, label="after", color="tab:blue")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("logit")
    ax.set_title(title or f"pred: {base['pred_label']} -> {after['pred_label']} | true: {base['true_label']}")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


def top1_repair_summary(repair_log: pd.DataFrame) -> pd.DataFrame:
    if repair_log.empty:
        return pd.DataFrame(columns=["edit_mode", "k", "n", "top1_repair_rate"])

    rows = []
    max_k = int(CONFIG["max_concepts_per_video"])
    log = repair_log.copy()
    log["num_annotations"] = log["num_annotations"].astype(int)
    log["repaired_top1"] = log["repaired_top1"].astype(bool)

    for edit_mode, mode_part in log.groupby("edit_mode"):
        sample_ids = sorted(mode_part["sample_idx"].astype(int).unique().tolist())
        first_success = {}
        for sample_idx, sample_part in mode_part.groupby("sample_idx"):
            sample_part = sample_part.sort_values("num_annotations")
            successes = sample_part[sample_part["repaired_top1"]]
            first_success[int(sample_idx)] = int(successes["num_annotations"].iloc[0]) if not successes.empty else None

        for k in range(1, max_k + 1):
            repaired = [first_success[s] is not None and first_success[s] <= k for s in sample_ids]
            rows.append({
                "edit_mode": edit_mode,
                "k": k,
                "n": len(sample_ids),
                "top1_repair_rate": float(np.mean(repaired)) if repaired else 0.0,
            })

    return pd.DataFrame(rows).sort_values(["edit_mode", "k"]).reset_index(drop=True)


REPAIR_LOG_ROWS = []
repair_log_df = pd.DataFrame()


def launch_sequential_repair_ui(
    candidates: pd.DataFrame,
    local_table: pd.DataFrame,
    global_table: pd.DataFrame,
    num_frames: int = 8,
    global_top_k: int = 10,
    log_csv: Optional[Path] = None,
):
    try:
        import ipywidgets as widgets
    except ModuleNotFoundError:
        print("ipywidgets is not installed in this notebook kernel.")
        print("Run this in a notebook cell, restart the kernel, then rerun this cell:")
        print("%pip install ipywidgets")
        return None

    if candidates.empty:
        print("No candidates available. Run candidate selection first.")
        return None

    if log_csv is None:
        log_csv = ROOT / CONFIG["manual_annotation_csv"]

    global REPAIR_LOG_ROWS, repair_log_df
    if log_csv.exists() and not REPAIR_LOG_ROWS:
        repair_log_df = pd.read_csv(log_csv)
        REPAIR_LOG_ROWS = repair_log_df.to_dict("records")

    candidate_records = candidates.to_dict("records")
    candidate_by_sample = {int(r["sample_idx"]): r for r in candidate_records}
    sample_ids = [int(r["sample_idx"]) for r in candidate_records]
    labels = [
        (
            f"{i + 1}/{len(sample_ids)} | sample {int(r['sample_idx'])}: "
            f"{r['pred_label']} -> {r['true_label']} (rank {int(r['true_rank'])})",
            int(r["sample_idx"]),
        )
        for i, r in enumerate(candidate_records)
    ]
    concept_options = [concept_label(i) for i in range(len(concept_names))]
    label_to_concept_idx = {label: i for i, label in enumerate(concept_options)}
    concept_cache = {}

    mode = widgets.ToggleButtons(options=[("Local", "local"), ("Global", "global")], value="local")
    dropdown = widgets.Dropdown(options=labels, value=sample_ids[0], description="sample")
    prev_button = widgets.Button(icon="arrow-left", tooltip="Previous sample", layout=widgets.Layout(width="44px"))
    next_button = widgets.Button(icon="arrow-right", tooltip="Next sample", layout=widgets.Layout(width="44px"))
    suggested_button = widgets.Button(description="Fill suggested", icon="magic", layout=widgets.Layout(width="130px"))
    test_button = widgets.Button(description="Test + log", icon="check", button_style="success", layout=widgets.Layout(width="120px"))
    skip_button = widgets.Button(description="Failure + next", icon="step-forward", layout=widgets.Layout(width="140px"))
    status = widgets.HTML()
    summary_output = widgets.Output()
    plot_output = widgets.Output()
    result_output = widgets.Output()
    editor_box = widgets.VBox()

    slot_widgets = []
    suppress_dropdown_callback = {"value": False}
    render_state = {"sample_idx": None, "rendering": False}

    def selected_sample_idx() -> int:
        return int(dropdown.value)

    def sample_position(sample_idx: int) -> int:
        return sample_ids.index(int(sample_idx))

    def concepts_t(sample_idx: int) -> torch.Tensor:
        if sample_idx not in concept_cache:
            concept_cache[sample_idx] = original_concepts_for_sample(cbm_model, sample_idx, device)
        return concept_cache[sample_idx]

    def parse_value(text: str):
        text = str(text).strip()
        if text == "":
            return np.nan
        return float(text)

    def original_value_text(sample_idx: int, edit_mode: str, window_idx: int, concept_idx: int) -> str:
        values = concepts_t(sample_idx)[:, int(concept_idx)]
        if edit_mode == "global":
            return f"orig mean={float(values.mean()):.4g}, range=[{float(values.min()):.4g}, {float(values.max()):.4g}]"
        w = min(max(int(window_idx), 0), int(values.numel()) - 1)
        return f"orig={float(values[w]):.4g}"

    def suggested_rows(sample_idx: int) -> pd.DataFrame:
        table = local_table if mode.value == "local" else global_table
        return table[table["sample_idx"].eq(sample_idx)].head(int(CONFIG["max_concepts_per_video"]))

    def refresh_original_labels(_=None):
        sample_idx = selected_sample_idx()
        max_window = int(concepts_t(sample_idx).shape[0]) - 1
        for controls in slot_widgets:
            controls["window"].max = max_window
            controls["window"].disabled = mode.value == "global"
            try:
                concept_idx = parse_concept_label(controls["concept"].value, label_to_concept_idx)
                controls["original"].value = original_value_text(
                    sample_idx,
                    mode.value,
                    controls["window"].value,
                    concept_idx,
                )
            except ValueError:
                controls["original"].value = "<span style='color:#b00020'>unknown concept</span>"

    def fill_suggested(_=None):
        rows = suggested_rows(selected_sample_idx())
        for i, controls in enumerate(slot_widgets):
            if i >= len(rows):
                controls["apply"].value = False
                controls["value"].value = ""
                continue
            row = rows.iloc[i]
            controls["apply"].value = True
            controls["concept"].value = concept_label(int(row["concept_idx"]))
            if mode.value == "local" and pd.notna(row["window_idx"]):
                controls["window"].value = int(row["window_idx"])
            suggested = row.get("suggested_correction_value", np.nan)
            controls["value"].value = "" if pd.isna(suggested) else f"{float(suggested):.6g}"
        refresh_original_labels()
        status.value = "Suggested slots filled. You can change concept, window, and target value before testing."

    def active_rows_from_widgets():
        rows = []
        sample_idx = selected_sample_idx()
        T, C = concepts_t(sample_idx).shape
        for slot_idx, controls in enumerate(slot_widgets, start=1):
            if not controls["apply"].value:
                continue
            try:
                value = parse_value(controls["value"].value)
            except ValueError:
                status.value = f"<span style='color:#b00020'>Slot {slot_idx} has a non-numeric target value.</span>"
                return None
            if pd.isna(value):
                status.value = f"<span style='color:#b00020'>Slot {slot_idx} is active but has no target value.</span>"
                return None
            try:
                concept_idx = parse_concept_label(controls["concept"].value, label_to_concept_idx)
            except ValueError as exc:
                status.value = f"<span style='color:#b00020'>Slot {slot_idx}: {exc}</span>"
                return None
            window_idx = int(controls["window"].value)
            if not 0 <= concept_idx < C:
                status.value = f"<span style='color:#b00020'>Slot {slot_idx} concept is outside range.</span>"
                return None
            if mode.value == "local" and not 0 <= window_idx < T:
                status.value = f"<span style='color:#b00020'>Slot {slot_idx} window is outside range 0..{T - 1}.</span>"
                return None
            original_values = concepts_t(sample_idx)[:, concept_idx]
            if mode.value == "global":
                original_value = float(original_values.mean())
                original_detail = f"mean={float(original_values.mean()):.6g}; min={float(original_values.min()):.6g}; max={float(original_values.max()):.6g}"
                window_for_log = np.nan
            else:
                original_value = float(original_values[window_idx])
                original_detail = f"w{window_idx}={original_value:.6g}"
                window_for_log = window_idx
            rows.append({
                "slot": slot_idx,
                "edit_mode": mode.value,
                "window_idx": window_for_log,
                "concept_idx": concept_idx,
                "concept_name": concept_names[concept_idx] if concept_idx < len(concept_names) else f"c{concept_idx}",
                "original_value": original_value,
                "original_detail": original_detail,
                "correction_value": float(value),
            })
        return rows[: int(CONFIG["max_concepts_per_video"])]

    def edits_from_rows(rows, k: int):
        prefix = rows[:k]
        if mode.value == "global":
            return [(int(row["concept_idx"]), float(row["correction_value"])) for row in prefix]
        return [(int(row["window_idx"]), int(row["concept_idx"]), float(row["correction_value"])) for row in prefix]

    def evaluate_prefix(sample_idx: int, edits):
        if mode.value == "global":
            return logits_with_global_concept_edits(cbm_model, sample_idx, edits, device) if edits else forward_logits(cbm_model, sample_idx, device)
        return logits_with_concept_edits(cbm_model, sample_idx, edits, device) if edits else forward_logits(cbm_model, sample_idx, device)

    def format_logit_report(sample_idx: int, k: int, rows, base_logits: torch.Tensor, after_logits: torch.Tensor, base: Dict[str, object], after: Dict[str, object]) -> str:
        class_ids = []
        for idx in [base["pred_idx"], base["true_idx"], after["pred_idx"]]:
            idx = int(idx)
            if idx not in class_ids:
                class_ids.append(idx)
        edited = "; ".join(
            f"c{int(row['concept_idx'])}={float(row['correction_value']):.6g}"
            if mode.value == "global"
            else f"w{int(row['window_idx'])}:c{int(row['concept_idx'])}={float(row['correction_value']):.6g}"
            for row in rows[:k]
        )
        lines = [
            f"sample {sample_idx} | {mode.value} intervention | k={k}",
            f"edited: {edited}",
            f"prediction: {base['pred_label']} -> {after['pred_label']} | true: {base['true_label']}",
            f"true rank: {int(base['true_rank'])} -> {int(after['true_rank'])}",
            "",
            "class logits:",
        ]
        for idx in class_ids:
            before = float(base_logits[idx].item())
            after_value = float(after_logits[idx].item())
            lines.append(f"  {label_name(cbm_model, idx)}: {before:.6g} -> {after_value:.6g} (delta {after_value - before:+.6g})")
        return "\n".join(lines)

    def append_log(row: Dict[str, object]):
        global repair_log_df
        key = (row["edit_mode"], int(row["sample_idx"]), int(row["num_annotations"]))
        REPAIR_LOG_ROWS[:] = [
            old for old in REPAIR_LOG_ROWS
            if (old.get("edit_mode"), int(old.get("sample_idx", -1)), int(old.get("num_annotations", -1))) != key
        ]
        REPAIR_LOG_ROWS.append(row)
        repair_log_df = pd.DataFrame(REPAIR_LOG_ROWS)
        repair_log_df.to_csv(log_csv, index=False)
        summary_output.clear_output(wait=True)
        with summary_output:
            display(top1_repair_summary(repair_log_df))

    def render_editor(sample_idx: int):
        slot_widgets.clear()
        T, _ = concepts_t(sample_idx).shape
        children = [widgets.HTML(f"<b>{mode.value.title()} annotations</b>: choose any concept and target value, up to {CONFIG['max_concepts_per_video']} slots.")]
        for slot_idx in range(1, int(CONFIG["max_concepts_per_video"]) + 1):
            apply = widgets.Checkbox(value=False, description="", indent=False, layout=widgets.Layout(width="30px"))
            window = widgets.BoundedIntText(value=0, min=0, max=max(0, T - 1), layout=widgets.Layout(width="70px"), disabled=mode.value == "global")
            concept = widgets.Combobox(
                options=concept_options,
                value=concept_options[0],
                ensure_option=False,
                placeholder="type concept name or c-id",
                layout=widgets.Layout(width="260px"),
            )
            value = widgets.Text(value="", placeholder="target", layout=widgets.Layout(width="95px"))
            original = widgets.HTML()
            controls = {"apply": apply, "window": window, "concept": concept, "value": value, "original": original}
            slot_widgets.append(controls)
            window.observe(refresh_original_labels, names="value")
            concept.observe(refresh_original_labels, names="value")
            children.append(widgets.HBox([
                widgets.HTML(f"<code>{slot_idx}</code>"),
                apply,
                widgets.HTML("w"),
                window,
                concept,
                widgets.HTML("to"),
                value,
                original,
            ]))
        editor_box.children = children
        fill_suggested()

    def render_sample(sample_idx: int, force: bool = False):
        sample_idx = int(sample_idx)
        if render_state["rendering"]:
            return
        if not force and render_state["sample_idx"] == sample_idx:
            return

        render_state["rendering"] = True
        try:
            result_output.clear_output(wait=True)
            render_editor(sample_idx)
            plot_output.clear_output(wait=True)
            with plot_output:
                inspect_video(
                    sample_idx,
                    local_table,
                    num_frames=num_frames,
                    show_global_concepts=True,
                    global_top_k=global_top_k,
                    window_top_k=global_top_k,
                    num_top_windows=4,
                )
            render_state["sample_idx"] = sample_idx
            pos = sample_position(sample_idx)
            status.value = f"Sample {pos + 1} / {len(sample_ids)} loaded for annotation."
        finally:
            render_state["rendering"] = False

    def navigate(delta: int):
        pos = sample_position(selected_sample_idx())
        new_pos = min(max(pos + delta, 0), len(sample_ids) - 1)
        suppress_dropdown_callback["value"] = True
        dropdown.value = sample_ids[new_pos]
        suppress_dropdown_callback["value"] = False
        render_sample(sample_ids[new_pos], force=True)

    def compact_edit_text(rows, key: str) -> str:
        return "; ".join(str(row[key]) for row in rows)

    def test_and_log(_=None):
        sample_idx = selected_sample_idx()
        rows = active_rows_from_widgets()
        if rows is None:
            return
        if not rows:
            status.value = "<span style='color:#b00020'>Activate at least one slot with a numeric target value.</span>"
            return

        base_logits = forward_logits(cbm_model, sample_idx, device)
        base = logits_to_record(cbm_model, sample_idx, base_logits)
        reports = []
        for k in range(1, len(rows) + 1):
            edits = edits_from_rows(rows, k)
            after_logits = evaluate_prefix(sample_idx, edits)
            after = logits_to_record(cbm_model, sample_idx, after_logits)
            repaired = bool(after["pred_idx"] == base["true_idx"])
            reports.append(format_logit_report(sample_idx, k, rows, base_logits, after_logits, base, after))
            result_output.clear_output(wait=True)
            with result_output:
                print("\n\n".join(reports))
            prefix = rows[:k]
            append_log({
                "edit_mode": mode.value,
                "sample_idx": sample_idx,
                "video_path": candidate_by_sample.get(sample_idx, {}).get("video_path"),
                "true_label": base["true_label"],
                "pred_before": base["pred_label"],
                "pred_after": after["pred_label"],
                "true_rank_before": int(base["true_rank"]),
                "true_rank_after": int(after["true_rank"]),
                "true_logit_before": float(base_logits[int(base["true_idx"])].item()),
                "true_logit_after": float(after_logits[int(base["true_idx"])].item()),
                "pred_before_logit_before": float(base_logits[int(base["pred_idx"])].item()),
                "pred_before_logit_after": float(after_logits[int(base["pred_idx"])].item()),
                "num_annotations": k,
                "repaired_top1": repaired,
                "edited_concepts": compact_edit_text(prefix, "concept_name"),
                "edited_slots": "; ".join(
                    f"c{int(row['concept_idx'])}" if mode.value == "global" else f"w{int(row['window_idx'])}:c{int(row['concept_idx'])}"
                    for row in prefix
                ),
                "original_values": compact_edit_text(prefix, "original_detail"),
                "target_values": compact_edit_text(prefix, "correction_value"),
            })
            if repaired:
                status.value = f"<span style='color:#0a7f28'>Repaired with k={k}. Logits shown below; click next when ready.</span>"
                return
        status.value = f"<span style='color:#9a6700'>Not repaired after {len(rows)} edits. Logits shown below; adjust values or log failure + next.</span>"

    def log_failure_and_next(_=None):
        sample_idx = selected_sample_idx()
        rows = active_rows_from_widgets()
        if rows is None:
            return
        k = max(1, len(rows))
        base = logits_to_record(cbm_model, sample_idx, forward_logits(cbm_model, sample_idx, device))
        append_log({
            "edit_mode": mode.value,
            "sample_idx": sample_idx,
            "video_path": candidate_by_sample.get(sample_idx, {}).get("video_path"),
            "true_label": base["true_label"],
            "pred_before": base["pred_label"],
            "pred_after": base["pred_label"],
            "true_rank_before": int(base["true_rank"]),
            "true_rank_after": int(base["true_rank"]),
            "num_annotations": k,
            "repaired_top1": False,
            "edited_concepts": compact_edit_text(rows, "concept_name"),
            "edited_slots": "; ".join(
                f"c{int(row['concept_idx'])}" if mode.value == "global" else f"w{int(row['window_idx'])}:c{int(row['concept_idx'])}"
                for row in rows
            ),
            "original_values": compact_edit_text(rows, "original_detail"),
            "target_values": compact_edit_text(rows, "correction_value"),
        })
        navigate(1)

    def on_dropdown_change(change):
        if change["name"] == "value" and not suppress_dropdown_callback["value"]:
            render_sample(int(change["new"]), force=True)


    def on_mode_change(change):
        if change["name"] == "value":
            render_editor(selected_sample_idx())
            status.value = f"Switched to {change['new']} edits for sample {selected_sample_idx()}."

    prev_button.on_click(lambda _: navigate(-1))
    next_button.on_click(lambda _: navigate(1))
    suggested_button.on_click(fill_suggested)
    test_button.on_click(test_and_log)
    skip_button.on_click(log_failure_and_next)
    dropdown.observe(on_dropdown_change, names="value")
    mode.observe(on_mode_change, names="value")

    controls = widgets.HBox([prev_button, dropdown, next_button])
    actions = widgets.HBox([mode, suggested_button, test_button, skip_button])
    ui = widgets.VBox([controls, actions, status, plot_output, editor_box, result_output, summary_output])
    display(ui)

    render_sample(sample_ids[0], force=True)

    return ui

from IPython.display import clear_output

clear_output(wait=True)
try:
    repair_ui.close()
except Exception:
    pass

repair_ui = launch_sequential_repair_ui(candidates_df, local_edit_df, global_edit_df)


## 8. Final Top-1 Repair-Rate Report

Run this after annotation. It summarizes the logged attempts by edit mode and number of annotations.


In [ ]:
if 'repair_log_df' not in globals() or repair_log_df.empty:
    log_path = ROOT / CONFIG["manual_annotation_csv"]
    if log_path.exists():
        repair_log_df = pd.read_csv(log_path)
    else:
        repair_log_df = pd.DataFrame()

repair_summary_df = top1_repair_summary(repair_log_df)
display(repair_summary_df)

if not repair_summary_df.empty:
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for edit_mode, part in repair_summary_df.groupby("edit_mode"):
        part = part.sort_values("k")
        ax.plot(part["k"], part["top1_repair_rate"], marker="o", label=edit_mode)
    ax.set_xlabel("Number of manual annotations")
    ax.set_ylabel("Top-1 repair rate")
    ax.set_xticks(range(1, int(CONFIG["max_concepts_per_video"]) + 1))
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()
